# 05 — Evidence contract, abstention and conflicts

## Why a contract is required

Retrieval scores only rank candidates. They cannot prove that an answer is complete. The evidence contract is deterministic code that checks every required business field and validates its source, entity, period and scope before allowing a claim.

## Status semantics

- `COMPLETE`: every required field has accepted evidence.
- `PARTIAL`: at least one field is supported and at least one required field is missing.
- `NOT_FOUND`: no required field has acceptable evidence; this is the correct answer for wrong scope/entity/period.
- `CONFLICT`: two accepted candidates support incompatible values for the same field. The system exposes both instead of choosing silently.

## Reading guide

The first experiment compares complete and deliberately truncated candidate sets. The second passes no candidates and must abstain. The third injects conflicting equity facts and must return `CONFLICT`. The last verifies that accepted evidence always carries a source locator, entity and excerpt. A passing result proves deterministic state transitions for these fixtures not that extraction is universally correct.

In [1]:
from pathlib import Path
from IPython.display import display
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table

ROOT = bootstrap()
from app.domain.models import AnswerStatus, EvidenceConstraints
from app.domain.profiles import explain_question_mapping, map_question
from app.evidence.gate import DeterministicEvidenceGate
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.planner import plan_queries
from app.store.artifacts import store

In [2]:
question = 'Quels indicateurs publics décrivent la position du Groupe en 2025 ?'
document_ids = ['foyer_financial_information_2025']
mapping_trace = explain_question_mapping(question)
display(display_table(mapping_trace))
profile = map_question(question)
print({'selected_profile': profile.id, 'required_fields': [field.id for field in profile.fields]})
queries = plan_queries(question, profile, document_ids)
retriever = HybridRetriever(store.selected_chunks(document_ids))
candidates_by_field = [retriever.search(query, k=5) for query in queries]
all_candidates = [candidate for batch in candidates_by_field for candidate in batch]
gate = DeterministicEvidenceGate()
complete_result = gate.evaluate(profile, all_candidates)
partial_result = gate.evaluate(profile, candidates_by_field[0])
assert complete_result.status == AnswerStatus.COMPLETE
assert partial_result.status == AnswerStatus.PARTIAL
{
    'complete': complete_result.status.value,
    'partial': partial_result.status.value,
    'partial_missing_fields': partial_result.missing_fields,
}

,profile_id,matched_triggers,score,selected,selection_note
0,prudential_coverage,[],0,False,
1,entity_prudential_coverage,[],0,False,
2,public_position,"[position, groupe]",4,True,highest trigger score (minimum 3)
3,customer_operations,[],0,False,
4,international_health,[],0,False,


{'selected_profile': 'public_position', 'required_fields': ['group_equity', 'non_life_market_share', 'business_areas']}


{'complete': 'COMPLETE',
 'partial': 'PARTIAL',
 'partial_missing_fields': ['Part de marché non-vie au Luxembourg',
  "Domaines d'activité"]}

In [3]:
empty_result = gate.evaluate(profile, [])
assert empty_result.status == AnswerStatus.NOT_FOUND
display_table([
    {'champ': coverage.field_id, 'statut': coverage.state}
    for coverage in empty_result.coverage
])

,champ,statut
0,group_equity,MISSING
1,non_life_market_share,MISSING
2,business_areas,MISSING


In [4]:
equity_candidate = next(
    candidate for candidate in all_candidates
    if candidate.field_id == 'group_equity'
    and any(fact.field_id == 'group_equity' for fact in candidate.chunk.facts)
)
original_fact = next(fact for fact in equity_candidate.chunk.facts if fact.field_id == 'group_equity')
conflicting_fact = original_fact.model_copy(update={'value': 999.0, 'formatted_value': '999,0 M€'})
conflicting_chunk = equity_candidate.chunk.model_copy(update={
    'id': f'{equity_candidate.chunk.id}-synthetic-conflict',
    'facts': [conflicting_fact],
})
conflicting_candidate = equity_candidate.model_copy(update={'chunk': conflicting_chunk})
conflict_result = gate.evaluate(profile, [*all_candidates, conflicting_candidate])
assert conflict_result.status == AnswerStatus.CONFLICT
display_table([
    {'champ': item.field_id, 'état': item.state, 'valeur': item.fact.formatted_value if item.fact else None}
    for item in conflict_result.evidence if item.field_id == 'group_equity'
])

,champ,état,valeur
0,group_equity,CONFLICT,"1 544,0 M€"
1,group_equity,CONFLICT,"1 544,0 M€"


In [5]:
accepted = [item for item in complete_result.evidence if item.state == 'ACCEPTED']
assert all(item.source and item.source.document_id for item in accepted)
assert all(item.fact and item.fact.entity for item in accepted)
assert all(item.excerpt for item in accepted)
{
    'accepted_evidence': len(accepted),
    'all_have_source': True,
    'all_have_entity': True,
    'all_have_excerpt': True,
}

{'accepted_evidence': 5,
 'all_have_source': True,
 'all_have_entity': True,
 'all_have_excerpt': True}

## Mapping is not entity extraction

The profile mapper reads only the question string (plus an optional explicit `profile_id`). It normalizes case/accents, matches complete trigger tokens, applies two reviewed priority rules, and selects the highest score if it reaches 3. `document_ids` do not select the profile. The output is a business contract: required fields and query templates.

Entity and period validation happens later. During offline ingestion, each typed fact receives `fact.entity` and `fact.period` from reviewed document metadata/table coordinates. At request time, `constraints.entity` and `constraints.period` are explicit inputs. The gate compares normalized equality for entity and controlled prefix equality for period. It does not infer an entity from a nearby sentence and does not ask Gemini to decide compatibility.

In the web UI, constraints are automatically sent only when exactly one selected document provides one unambiguous entity and year. With several documents, no entity/period constraint is silently invented. A future NER/LLM extractor could propose constraints, but it would require confirmation and evaluation before being trusted.

In [6]:
fact_rows = []
for item in complete_result.evidence:
    if item.fact and item.source:
        fact_rows.append({
            'field': item.field_id,
            'value': item.fact.formatted_value,
            'entity_from_ingestion': item.fact.entity,
            'period_from_ingestion': item.fact.period,
            'document_id': item.source.document_id,
            'page': item.source.page,
            'state': item.state,
        })
display(display_table(fact_rows))

,field,value,entity_from_ingestion,period_from_ingestion,document_id,page,state
0,group_equity,"1 544,0 M€",Groupe Foyer,2025-12-31,foyer_financial_information_2025,1,ACCEPTED
1,group_equity,"1 544,0 M€",Groupe Foyer,2025-12-31,foyer_financial_information_2025,1,ACCEPTED
2,non_life_market_share,40 %,Groupe Foyer,2025,foyer_financial_information_2025,1,ACCEPTED
3,non_life_market_share,40 %,Groupe Foyer,2025,foyer_financial_information_2025,1,ACCEPTED
4,business_areas,"3 domaines : non-vie, vie et gestion patrimoniale",Groupe Foyer,2025,foyer_financial_information_2025,1,ACCEPTED


## Expanded decision matrix

The next experiment runs the same candidates through six controlled scenarios. `COMPLETE` and `PARTIAL` are both valid outcomes: `PARTIAL` is preferable to fabricating missing fields. `NOT_FOUND` means all required fields are missing or rejected. `CONFLICT` has priority when incompatible accepted values exist for the same entity, field and period.

In [7]:
wrong_entity = gate.evaluate(
    profile, all_candidates, EvidenceConstraints(entity='Another Insurance Group')
)
wrong_period = gate.evaluate(
    profile, all_candidates, EvidenceConstraints(period='2024')
)
cases = {
    'all required fields supported': complete_result,
    'only one retrieval batch retained': partial_result,
    'no candidates': empty_result,
    'wrong requested entity': wrong_entity,
    'wrong requested period': wrong_period,
    'incompatible value injected': conflict_result,
}
decision_rows = []
for name, result in cases.items():
    decision_rows.append({
        'scenario': name,
        'answer_status': result.status.value,
        'covered': ', '.join(item.field_id for item in result.coverage if item.state != 'MISSING'),
        'missing': ', '.join(result.missing_fields),
        'rejected_reasons': ' | '.join(
            item.reason or '' for item in result.evidence if item.state == 'REJECTED'
        ),
    })
assert wrong_entity.status == AnswerStatus.NOT_FOUND
assert wrong_period.status == AnswerStatus.NOT_FOUND
display(display_table(decision_rows))

,scenario,answer_status,covered,missing,rejected_reasons
0,all required fields supported,COMPLETE,"group_equity, non_life_market_share, business_...",,
1,only one retrieval batch retained,PARTIAL,group_equity,"Part de marché non-vie au Luxembourg, Domaines...",
2,no candidates,NOT_FOUND,,"Capitaux propres part du Groupe, Part de march...",
3,wrong requested entity,NOT_FOUND,,"Capitaux propres part du Groupe, Part de march...",Preuve incompatible avec le contrat : entité '...
4,wrong requested period,NOT_FOUND,,"Capitaux propres part du Groupe, Part de march...",Preuve incompatible avec le contrat : période ...
5,incompatible value injected,CONFLICT,"group_equity, non_life_market_share, business_...",,


# Profil public_position
# │
# ├── group_equity
# │   └── preuve trouvée
# │
# ├── non_life_market_share
# │   └── preuve absente du lot partiel
# │
# └── business_areas
#     └── preuve absente du lot partiel

## Robust mapping with bounded model use

The production mapper is a cascade, not an LLM call on every question:

1. Canonical entity aliases and years are extracted deterministically.
2. Reviewed lexical triggers select clear intents with zero model calls.
3. Only an unclear intent is embedded against reviewed profile prototypes.
4. A profile is accepted from dense scores only with sufficient score and margin.
5. Otherwise one structured Gemini judge chooses among the top three or returns null.
6. The decision is cached by normalized question/profile/constraints.
7. Provider failure or unresolved ambiguity returns `open_question`; it never guesses.

The model selects an intent contract only. It cannot accept evidence or override entity/period compatibility. The following paraphrase set makes actual model use and abstention visible. Set `ENABLE_HYBRID_MAPPING=1` before starting the kernel to run the paid ambiguous branch.

In [8]:
from app.domain.mapping import HybridQuestionMapper

mapping_cases = [
    ('clear_rule', 'What is FGH SCR coverage in 2025?', 'entity_prudential_coverage'),
    ('financial_paraphrase', 'How strong is the capital cushion and competitive footprint of the insurer?', 'public_position'),
    ('customer_paraphrase', 'How do policyholders perceive the digital service and claims journey?', 'customer_operations'),
    ('genuinely_vague', 'Tell me something interesting about the company', 'open_question'),
]
hybrid_mapper = HybridQuestionMapper()
mapping_rows = []
for case, text, expected in mapping_cases:
    mapped = hybrid_mapper.map(text)
    mapping_rows.append({
        'case': case,
        'expected': expected,
        'obtained': mapped.profile.id,
        'correct': mapped.profile.id == expected,
        'source': mapped.decision.decision_source,
        'confidence': mapped.decision.confidence,
        'entity': mapped.constraints.entity,
        'period': mapped.constraints.period,
        'model_calls': ', '.join(
            f'{call.purpose}:{call.status}' for call in mapped.decision.model_calls
        ) or 'none',
    })
display(display_table(mapping_rows))

,case,expected,obtained,correct,source,confidence,entity,period,model_calls
0,clear_rule,entity_prudential_coverage,entity_prudential_coverage,True,rules,1.0,Foyer Global Health S.A.,2025,none
1,financial_paraphrase,public_position,open_question,False,abstention,0.0,None,None,ambiguous_profile_embedding:ERROR
2,customer_paraphrase,customer_operations,open_question,False,abstention,0.0,None,None,ambiguous_profile_embedding:ERROR
3,genuinely_vague,open_question,open_question,True,abstention,0.0,None,None,ambiguous_profile_embedding:ERROR


In [9]:
cached_first = hybrid_mapper.map(mapping_cases[1][1])
cached_second = hybrid_mapper.map(mapping_cases[1][1])
assert cached_first is cached_second
{
    'same_in_memory_decision': cached_first is cached_second,
    'additional_model_calls_for_identical_question': 0,
}

{'same_in_memory_decision': True,
 'additional_model_calls_for_identical_question': 0}